# Notebook 09 — Supplementary Table: Country-Level Predictor Values (H3)

Generates Table S[N]: Country-level predictor values used in H3 cross-national analysis.
Exports to DOCX for inclusion in Supplementary Materials.

OSF Amendment 9 (2026-05-XX): Added to comply with STROBE item 14a.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

DATA_DIR       = Path('../data')
PREDICTORS_CSV = DATA_DIR / 'predictors' / 'H3_predictors.csv'
OUTPUT_DIR     = Path('../doc/supplementary')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Predictors CSV: {PREDICTORS_CSV}')
print(f'Output dir:     {OUTPUT_DIR}')

Predictors CSV: ..\data\predictors\H3_predictors.csv
Output dir:     ..\doc\supplementary


In [2]:
# ── Load predictors ───────────────────────────────────────────────────────────
df = pd.read_csv(PREDICTORS_CSV)
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head()

Loaded: 21 rows × 6 columns
Columns: ['country_code', 'country_name', 'who_atlas_score', 'gdp_per_capita', 'sci_output_psych', 'pew_religiosity']


,country_code,country_name,who_atlas_score,gdp_per_capita,sci_output_psych,pew_religiosity
0,UA,Ukraine,NaN,5140.0,8.00,84.8
1,DE,Germany,13.1,54777.0,73.20,63.9
2,FR,France,14.5,44700.0,43.71,57.4
3,PL,Poland,3.5,22145.0,33.21,91.4
4,NL,Netherlands,NaN,63516.0,187.65,45.9


In [3]:
# ── Country names mapping ─────────────────────────────────────────────────────
# Adjust codes to match your CSV if needed
COUNTRY_NAMES = {
    'UA': 'Ukraine',
    'DE': 'Germany',
    'FR': 'France',
    'PL': 'Poland',
    'NL': 'Netherlands',
    'LV': 'Latvia',
    'LT': 'Lithuania',
    'EE': 'Estonia',
    'US': 'United States',
    'GB': 'United Kingdom',
    'AU': 'Australia',
    'CA': 'Canada',
    'GE': 'Georgia',
    'AM': 'Armenia',
    'AZ': 'Azerbaijan',
    'KZ': 'Kazakhstan',
    'UZ': 'Uzbekistan',
    'MD': 'Moldova',
    'IL': 'Israel',
    'JP': 'Japan',
    'BR': 'Brazil',
}

# ── Identify column names in your CSV ─────────────────────────────────────────
# Print actual column names so we can map them correctly
print('Actual columns in CSV:')
for col in df.columns:
    print(f'  {col}')

Actual columns in CSV:
  country_code
  country_name
  who_atlas_score
  gdp_per_capita
  sci_output_psych
  pew_religiosity


In [4]:
# ── Build display table ───────────────────────────────────────────────────────
# IMPORTANT: Adjust column names below to match your actual CSV columns
# Common patterns: 'country_code', 'gdp_per_capita', 'sci_output', 'religiosity', 'who_atlas'
# Run the cell above first to see actual names

# --- EDIT THESE TO MATCH YOUR CSV COLUMNS ---
COL_CODE        = 'country_code'     # country ISO code column
COL_GDP         = 'gdp_per_capita'   # GDP per capita (USD, World Bank 2023)
COL_SCI         = 'sci_output_psych'       # Psychology publications per 1M pop (SCImago 2023)
COL_REL         = 'pew_religiosity'      # Religiosity index (Pew Research 2020)
COL_WHO         = 'who_atlas_score'        # Mental health expenditure (WHO Atlas 2020) — may have NaN
# --------------------------------------------

tbl = df[[COL_CODE, 'country_name', COL_GDP, COL_SCI, COL_REL, COL_WHO]].copy()
tbl['Country'] = tbl['country_name']

# Sort: Ukraine first (focal), then alphabetically
tbl['_sort'] = tbl[COL_CODE].apply(lambda x: '0' if x == 'UA' else x)
tbl = tbl.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

# Format columns
tbl['GDP per capita (USD)'] = tbl[COL_GDP].apply(
    lambda x: f'{x:,.0f}' if pd.notna(x) else 'N/A')
tbl['Psychology publications\nper 1M population'] = tbl[COL_SCI].apply(
    lambda x: f'{x:.1f}' if pd.notna(x) else 'N/A')
tbl['Religiosity index'] = tbl[COL_REL].apply(
    lambda x: f'{x:.3f}' if pd.notna(x) else 'N/A')
tbl['MH expenditure\n(% of health budget)'] = tbl[COL_WHO].apply(
    lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A\u1d43')

display_cols = [
    'Country',
    'GDP per capita (USD)',
    'Psychology publications\nper 1M population',
    'Religiosity index',
    'MH expenditure\n(% of health budget)',
]
tbl_display = tbl[display_cols].copy()
tbl_display.index = range(1, len(tbl_display) + 1)
print(tbl_display.to_string())
print(f'\nWHO Atlas available: {tbl[COL_WHO].notna().sum()}/21 countries')

           Country GDP per capita (USD) Psychology publications\nper 1M population Religiosity index MH expenditure\n(% of health budget)
1          Ukraine                5,140                                        8.0            84.800                                 N/Aᵃ
2          Armenia                8,159                                        7.1            99.000                                 2.30
3        Australia               65,058                                      185.8            57.700                                 N/Aᵃ
4       Azerbaijan                7,133                                        1.3            95.300                                 3.50
5           Brazil               10,378                                        8.7            86.500                                 1.60
6           Canada               54,220                                      136.1            65.400                                 N/Aᵃ
7          Germany               5

In [5]:
# ── Export to DOCX via python-docx ───────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'python-docx'], check=True, capture_output=True)

from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

doc = Document()

# Поля сторінки
section = doc.sections[0]
section.top_margin    = Cm(2.54)
section.bottom_margin = Cm(2.54)
section.left_margin   = Cm(2.54)
section.right_margin  = Cm(2.54)

# Заголовок таблиці
title_p = doc.add_paragraph()
title_r = title_p.add_run(
    'Table S[N]. Country-level predictor values used in H3 cross-national analysis.')
title_r.bold = True
title_r.font.size = Pt(10)
title_r.font.name = 'Times New Roman'
title_p.paragraph_format.space_after = Pt(6)

# Заголовки колонок
col_headers = [
    'Country',
    'GDP per capita (USD)\n(World Bank, 2023)',
    'Psychology publications\nper 1M population\n(SCImago, 2023)',
    'Religiosity index\n(Pew Research, 2020)',
    'MH expenditure\n(% of health budget)\n(WHO Atlas, 2020)',
]

# Таблиця
table = doc.add_table(rows=1, cols=5)
table.style = 'Table Grid'

# Ширини колонок (у см, сума ≈ 16 см)
col_widths = [Cm(3.5), Cm(3.0), Cm(3.5), Cm(3.0), Cm(3.0)]

# Заголовний рядок
hdr_cells = table.rows[0].cells
for i, (cell, header) in enumerate(zip(hdr_cells, col_headers)):
    cell.width = col_widths[i]
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(header)
    run.bold = True
    run.font.size = Pt(9)
    run.font.name = 'Times New Roman'
    # Блакитний фон заголовка
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), 'DCE6F1')
    tcPr.append(shd)

# Рядки даних
for _, row in tbl_display.iterrows():
    row_cells = table.add_row().cells
    for i, (cell, value) in enumerate(zip(row_cells, row.values)):
        cell.width = col_widths[i]
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT if i == 0 else WD_ALIGN_PARAGRAPH.CENTER
        run = p.add_run(str(value))
        run.font.size = Pt(9)
        run.font.name = 'Times New Roman'

# Виноска
fn_p = doc.add_paragraph()
fn_p.paragraph_format.space_before = Pt(4)
fn_run = fn_p.add_run(
    '\u1d43 WHO Mental Health Atlas 2020 data available for 9 of 21 countries only; '
    'N/A indicates data not available. MH = mental health.')
fn_run.italic = True
fn_run.font.size = Pt(8)
fn_run.font.name = 'Times New Roman'

# Зберегти
output_path = Path('../doc/supplementary/table_S_predictors.docx')
output_path.parent.mkdir(parents=True, exist_ok=True)
doc.save(output_path)
print(f'✅ Saved: {output_path}')

✅ Saved: ..\doc\supplementary\table_S_predictors.docx


## Done

File saved to `doc/supplementary/table_S_predictors.docx`.

Next steps:
1. Open the DOCX and verify formatting
2. Insert into Supplementary Materials document
3. Update OSF Amendment 9
4. Update STROBE checklist item 14a with page number